# Swin-FANE ARWM Colab Experiments

**Project:** A Low-Latency Video Analytics Framework for Emotion Recognition and Stress Inference Using Region-Aware Swin-FANE and Temporal Modelling.

This notebook prepares a controlled FER2013 comparison of the existing `off`, `fixed`, and `adaptive` ARWM modes. It does **not** implement the proposed Context-Aware Adaptive Facial Region Weighting enhancement. The training section is deliberately approval-gated and is not launched by default. Use a Colab GPU runtime. Every variant uses a separate Google Drive checkpoint/log/results directory; test data is reserved for final evaluation only.

## 1. Environment and GPU verification

Run this first. It stops immediately if CUDA is unavailable. It records the installed PyTorch/torchvision/CUDA versions, GPU model, and free/total VRAM; it does not install or upgrade packages.

In [ ]:
import json, os, platform, subprocess, sys, time
from pathlib import Path
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. In Colab select Runtime > Change runtime type > T4 GPU (or another GPU), reconnect, and rerun from this cell. No training has started.')
device = torch.device('cuda:0')
torch.cuda.set_device(device)
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
ENVIRONMENT = {
    'python': sys.version,
    'platform': platform.platform(),
    'pytorch': torch.__version__,
    'torchvision': torchvision.__version__,
    'torch_cuda_build': torch.version.cuda,
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(device),
    'gpu_total_memory_bytes': int(total_bytes),
    'gpu_free_memory_bytes': int(free_bytes),
    'gpu_capability': torch.cuda.get_device_capability(device),
}
print(json.dumps(ENVIRONMENT, indent=2, default=str))

## 2. Google Drive mounting and project backup

Create a **sanitized source ZIP on Windows** and place it in Drive or upload it when prompted in the next section. Include project Python/YAML/tests, `requirements.txt`, `README.md`, `results/arwm/pre_change_baseline.json`, and the two original checkpoints only. Do not include `.venv*`, `.git`, datasets, videos, caches, `.env`, credentials, or logs. The original Windows directory is never extracted over or changed.

PowerShell source-ZIP recipe (run on Windows from the project directory; it refuses to overwrite the ZIP):

```powershell
$root = (Resolve-Path 'C:\Users\AMSV\OneDrive\Desktop\SWIN').Path
$zipPath = Join-Path $root 'SWIN_ARWM_Colab_source.zip'
if (Test-Path $zipPath) { throw "Archive already exists: $zipPath" }
Add-Type -AssemblyName System.IO.Compression
$files = @()
foreach ($folder in @('models','data','utils','scripts','tests')) {
  $files += Get-ChildItem -LiteralPath (Join-Path $root $folder) -File -Recurse | Where-Object { $_.Extension -eq '.py' }
}
$files += Get-ChildItem -LiteralPath (Join-Path $root 'configs') -File -Recurse | Where-Object { $_.Extension -eq '.yaml' }
foreach ($relative in @('README.md','requirements.txt','results\arwm\pre_change_baseline.json','checkpoints\fer2013_weighted_freeze3_lr1e4\best_model.pth','checkpoints\fer2013_arwm\best_model.pth')) {
  $path = Join-Path $root $relative
  if (-not (Test-Path -LiteralPath $path -PathType Leaf)) { throw "Required source file missing: $path" }
  $files += Get-Item -LiteralPath $path
}
$stream = [System.IO.File]::Open($zipPath,[System.IO.FileMode]::CreateNew)
$archive = [System.IO.Compression.ZipArchive]::new($stream,[System.IO.Compression.ZipArchiveMode]::Create)
try {
  foreach ($file in ($files | Sort-Object FullName -Unique)) {
    $entryName = $file.FullName.Substring($root.Length).TrimStart('\') -replace '\\','/'
    $entry = $archive.CreateEntry($entryName,[System.IO.Compression.CompressionLevel]::Optimal)
    $source = [System.IO.File]::OpenRead($file.FullName); $target = $entry.Open()
    try { $source.CopyTo($target) } finally { $target.Dispose(); $source.Dispose() }
  }
} finally { $archive.Dispose(); $stream.Dispose() }
Get-FileHash -Algorithm SHA256 (Join-Path $root 'checkpoints\fer2013_weighted_freeze3_lr1e4\best_model.pth'),(Join-Path $root 'checkpoints\fer2013_arwm\best_model.pth')
Write-Output "Created sanitized archive: $zipPath"
```

The notebook verifies the archive contents and known checkpoint hashes before using it.

## 3. Project upload or extraction

The archive is copied to Drive as a durable project backup and safely extracted to a new temporary Colab directory. A unique `RUN_ID` is created on first use. After a Colab disconnect, set `RUN_ID` to that same recorded value and set `RESUME_EXISTING_RUN = True` before continuing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
import hashlib, shutil, zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest().upper()

DRIVE_ROOT = Path('/content/drive/MyDrive/SWIN_ARWM_Colab')
BACKUP_DIR = DRIVE_ROOT / 'project_backups'
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = None  # On recovery, replace with the existing recorded run ID.
RESUME_EXISTING_RUN = False
if RUN_ID is None:
    RUN_ID = time.strftime('run_%Y%m%d_%H%M%S')
RUN_ROOT = DRIVE_ROOT / 'experiments' / RUN_ID
if RUN_ROOT.exists() and any(RUN_ROOT.iterdir()) and not RESUME_EXISTING_RUN:
    raise FileExistsError(f'{RUN_ROOT} already contains an experiment. Set RESUME_EXISTING_RUN=True only to continue that run, or choose a new RUN_ID.')
RUN_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT_ARCHIVE_DRIVE = None  # Optional existing Drive path to the sanitized ZIP.
if PROJECT_ARCHIVE_DRIVE and Path(PROJECT_ARCHIVE_DRIVE).is_file():
    source_path = Path(PROJECT_ARCHIVE_DRIVE)
    backup_path = BACKUP_DIR / source_path.name
    if source_path.resolve() != backup_path.resolve():
        if backup_path.exists():
            if sha256_file(source_path) != sha256_file(backup_path):
                raise FileExistsError(f'Drive backup exists with different contents: {backup_path}')
        else:
            shutil.copy2(source_path, backup_path)
    archive_source = backup_path
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one sanitized project ZIP.')
    upload_name = next(iter(uploaded))
    archive_source = Path('/content') / upload_name
    backup_path = BACKUP_DIR / upload_name
    if backup_path.exists():
        digest = sha256_file(archive_source)[:12]
        backup_path = BACKUP_DIR / f'{Path(upload_name).stem}_{digest}{Path(upload_name).suffix}'
    shutil.copy2(archive_source, backup_path)
    archive_source = backup_path
PROJECT_DIR = Path('/content') / f'SWIN_ARWM_project_{RUN_ID}'
if PROJECT_DIR.exists():
    raise FileExistsError(f'Temporary project path already exists: {PROJECT_DIR}')
PROJECT_DIR.mkdir(parents=True)
print({'run_id': RUN_ID, 'run_root': str(RUN_ROOT), 'source_backup_zip': str(archive_source), 'project_dir': str(PROJECT_DIR)})

## 4. Dependency installation

Install from the project's `requirements.txt`. Constraints preserve the runtime's already-installed PyTorch, torchvision, and timm versions; any of those packages that are absent are installed from the project requirements. If dependencies cannot be resolved without replacing already-installed core versions, installation stops instead of silently changing the model stack.

In [ ]:
import zipfile
with zipfile.ZipFile(archive_source) as zf:
    names = zf.namelist()
    forbidden = ('.venv/', '.git/', '__pycache__/', '.env', 'data/FER2013/', 'data/videos/')
    bad = [name for name in names if any(token in name for token in forbidden)]
    if bad:
        raise ValueError(f'Source archive includes excluded files: {bad[:10]}')
    required = {
        'models/swin_fane.py', 'models/full_framework.py',
        'data/fer2013_dataset.py', 'data/augmentations.py',
        'utils/configuration.py', 'utils/checkpointing.py',
        'scripts/train.py', 'scripts/evaluate_arwm_variants.py',
        'tests/test_arwm_modes.py', 'requirements.txt',
        'configs/arwm/off.yaml', 'configs/arwm/fixed.yaml',
        'configs/arwm/adaptive.yaml',
        'checkpoints/fer2013_weighted_freeze3_lr1e4/best_model.pth',
        'checkpoints/fer2013_arwm/best_model.pth',
    }
    missing = sorted(required - set(names))
    if missing:
        raise FileNotFoundError(f'Source archive is missing required files: {missing}')
    for info in zf.infolist():
        destination = (PROJECT_DIR / info.filename).resolve()
        if PROJECT_DIR.resolve() not in destination.parents and destination != PROJECT_DIR.resolve():
            raise ValueError(f'Unsafe ZIP entry path: {info.filename}')
    zf.extractall(PROJECT_DIR)

sys.path.insert(0, str(PROJECT_DIR))
ORIGINAL_CHECKPOINTS = {
    'baseline': PROJECT_DIR / 'checkpoints/fer2013_weighted_freeze3_lr1e4/best_model.pth',
    'adaptive_legacy': PROJECT_DIR / 'checkpoints/fer2013_arwm/best_model.pth',
}
EXPECTED_ORIGINAL_SHA256 = {
    'baseline': '0CDF99EF322ACBF835A34F46EE46E8DDD63899361DC266E8C624F848FF105F7A',
    'adaptive_legacy': '1BF4E5C2E071E1BAD4ADE8889AF3EC8DA980EF0E9A5808C2967DCEF13F5CB0F3',
}
CHECKPOINT_HASHES_BEFORE = {name: sha256_file(path) for name, path in ORIGINAL_CHECKPOINTS.items()}
if CHECKPOINT_HASHES_BEFORE != EXPECTED_ORIGINAL_SHA256:
    raise RuntimeError(f'Original checkpoint hash mismatch; do not proceed: {CHECKPOINT_HASHES_BEFORE}')
print('Original checkpoint hashes verified:', CHECKPOINT_HASHES_BEFORE)

In [ ]:
import importlib.metadata as package_metadata
core_packages = ('torch', 'torchvision', 'timm')
versions_before = {}
for name in core_packages:
    try:
        versions_before[name] = package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        pass  # requirements.txt installs absent packages; existing versions are pinned below.
constraints = Path('/content/swin_arwm_constraints.txt')
constraints.write_text(''.join(f'{name}=={version.split("+")[0]}\n' for name, version in versions_before.items()))
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt'), '-c', str(constraints)],
    text=True, capture_output=True,
)
print(install.stdout[-6000:])
if install.returncode:
    print(install.stderr[-6000:])
    raise RuntimeError('Project requirements could not be installed while preserving core model package versions.')
versions_after = {name: package_metadata.version(name) for name in core_packages}
changed_existing = {name: (before, versions_after.get(name)) for name, before in versions_before.items() if versions_after.get(name) != before}
if changed_existing:
    raise RuntimeError(f'Existing core model dependencies changed: {changed_existing}')
import albumentations, cv2, timm, yaml, sklearn
ENVIRONMENT.update({
    'core_versions_before_install': versions_before,
    'core_versions_after_install': versions_after,
    'numpy': package_metadata.version('numpy'),
    'timm': timm.__version__,
    'albumentations': albumentations.__version__,
    'opencv': cv2.__version__,
})
print('Required imports passed; core versions unchanged.')

## 5. Repository, test, and configuration verification

This checks the real filenames and CLI options, runs the existing ARWM unit suite, loads the three inherited YAML configurations, and compares their resolved data/augmentation/training/evaluation settings. It fails before training on any unapproved mismatch.

In [ ]:
from utils.configuration import load_config, get_region_attention_mode
from utils.checkpointing import validate_checkpoint_region_mode
from models.full_framework import build_framework
from data.fer2013_dataset import build_fer2013_dataloaders, FER2013_CLASSES, FER2013_FOLDER_TO_LABEL
from scripts.train import compute_inverse_frequency_weights, set_seed

required_paths = [
    'models/swin_fane.py', 'models/full_framework.py', 'models/swin_transformer.py',
    'data/fer2013_dataset.py', 'data/augmentations.py', 'utils/configuration.py',
    'utils/checkpointing.py', 'scripts/train.py', 'scripts/evaluate.py',
    'scripts/evaluate_arwm_variants.py', 'scripts/benchmark_inference.py',
    'tests/test_arwm_modes.py', 'configs/arwm/off.yaml',
    'configs/arwm/fixed.yaml', 'configs/arwm/adaptive.yaml',
]
missing = [rel for rel in required_paths if not (PROJECT_DIR / rel).is_file()]
if missing:
    raise FileNotFoundError(missing)
test_run = subprocess.run(
    [sys.executable, '-m', 'unittest', 'tests.test_arwm_modes', '-v'],
    cwd=PROJECT_DIR, text=True, capture_output=True,
)
print(test_run.stdout[-6000:])
if test_run.returncode:
    print(test_run.stderr[-6000:])
    raise RuntimeError('ARWM unit tests failed; do not train.')

MODES = ('off', 'fixed', 'adaptive')
CONFIGS = {mode: load_config(str(PROJECT_DIR / f'configs/arwm/{mode}.yaml')) for mode in MODES}
for mode, config in CONFIGS.items():
    if get_region_attention_mode(config) != mode:
        raise AssertionError(f'{mode} config resolved to the wrong mode')
    if config['training']['seed'] != 42 or config['training']['batch_size'] != 32 or config['training']['num_epochs'] != 50:
        raise AssertionError(f'Unexpected verified training settings for {mode}')
    if config['training']['optimizer'] != 'AdamW' or config['training']['learning_rate'] != 1e-4 or config['training']['scheduler'] != 'StepLR' or config['training']['freeze_backbone_epochs'] != 3:
        raise AssertionError(f'Unexpected optimizer/schedule settings for {mode}')

def parity_view(config):
    import copy
    value = copy.deepcopy(config)
    region = value['model']['region_attention']
    region.pop('mode', None)
    region.pop('enabled', None)
    value.pop('paths', None)
    return value
base_view = parity_view(CONFIGS['off'])
if any(parity_view(CONFIGS[mode]) != base_view for mode in ('fixed', 'adaptive')):
    raise AssertionError('Resolved variants differ beyond intended ARWM mode/output-path settings')
assert FER2013_FOLDER_TO_LABEL == {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'sad': 4, 'surprise': 5, 'neutral': 6}
print('Config parity verified:', {mode: {'seed': CONFIGS[mode]['training']['seed'], 'batch': CONFIGS[mode]['training']['batch_size'], 'epochs': CONFIGS[mode]['training']['num_epochs'], 'mode': get_region_attention_mode(CONFIGS[mode])} for mode in MODES})

## 6. FER2013 dataset preparation

Upload the official/extracted dataset to Drive separately. The notebook copies or safely extracts it to temporary Colab storage for faster training. No dataset is included in the source archive. Set `DATA_SOURCE` to a Drive folder (`FER2013/train` and `FER2013/test`) or a ZIP containing that layout.

In [ ]:
DATA_SOURCE = '/content/drive/MyDrive/FER2013'  # Set this to your actual Drive folder or ZIP.
DATA_ROOT = Path('/content/FER2013')
if not Path(DATA_SOURCE).exists():
    raise FileNotFoundError(f'Dataset source not found: {DATA_SOURCE}')
if DATA_ROOT.exists():
    raise FileExistsError(f'Temporary dataset path already exists: {DATA_ROOT}')
source_path = Path(DATA_SOURCE)
if source_path.is_dir():
    shutil.copytree(source_path, DATA_ROOT)
elif source_path.suffix.lower() == '.zip':
    staging = Path('/content/fer2013_extract')
    staging.mkdir()
    with zipfile.ZipFile(source_path) as zf:
        for info in zf.infolist():
            target = (staging / info.filename).resolve()
            if staging.resolve() not in target.parents and target != staging.resolve():
                raise ValueError(f'Unsafe dataset ZIP entry: {info.filename}')
        zf.extractall(staging)
    candidates = [staging] + [p for p in staging.rglob('*') if p.is_dir()]
    roots = [p for p in candidates if (p / 'train').is_dir() and (p / 'test').is_dir()]
    if len(roots) != 1:
        raise RuntimeError(f'Expected one FER2013 root with train/test, found: {roots}')
    shutil.copytree(roots[0], DATA_ROOT)
else:
    raise ValueError('DATA_SOURCE must be a dataset folder or ZIP file')
print('Local dataset root:', DATA_ROOT)

## 7. Dataset count and class distribution validation

The expected loader split is 24,403 train / 4,306 validation / 7,178 official test. Validation is a deterministic stratified 15% held out from the 28,709-image train folder using seed 42. The test folder remains untouched for final evaluation. Counts and split-manifest hashes are recorded to Drive.

In [ ]:
EXPECTED_COUNTS = {'train': 24403, 'validation': 4306, 'official_test': 7178}
EXPECTED_CLASS_COUNTS = {
    'train': {'anger': 3396, 'disgust': 371, 'fear': 3482, 'happiness': 6133, 'sadness': 4106, 'surprise': 2695, 'neutral': 4220},
    'validation': {'anger': 599, 'disgust': 65, 'fear': 615, 'happiness': 1082, 'sadness': 724, 'surprise': 476, 'neutral': 745},
    'official_test': {'anger': 958, 'disgust': 111, 'fear': 1024, 'happiness': 1774, 'sadness': 1247, 'surprise': 831, 'neutral': 1233},
}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
disk_counts = {}
expected_disk_counts = {
    'train': {'angry': 3995, 'disgust': 436, 'fear': 4097, 'happy': 7215, 'sad': 4830, 'surprise': 3171, 'neutral': 4965},
    'test': {'angry': 958, 'disgust': 111, 'fear': 1024, 'happy': 1774, 'sad': 1247, 'surprise': 831, 'neutral': 1233},
}
for split in ('train', 'test'):
    split_path = DATA_ROOT / split
    if not split_path.is_dir():
        raise FileNotFoundError(split_path)
    disk_counts[split] = {
        class_name: sum(1 for path in (split_path / class_name).iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
        for class_name in FER2013_FOLDER_TO_LABEL
    }
    if not all((split_path / class_name).is_dir() for class_name in FER2013_FOLDER_TO_LABEL):
        raise FileNotFoundError(f'Missing one or more expected class directories in {split_path}')
    if disk_counts[split] != expected_disk_counts[split]:
        raise AssertionError(f'{split} per-class image counts changed: {disk_counts[split]}')

train_loader, val_loader, test_loader = build_fer2013_dataloaders(
    root_dir=str(DATA_ROOT), image_size=CONFIGS['off']['dataset']['image_size'],
    batch_size=32, num_workers=0, pin_memory=False, seed=42,
)
loaders = {'train': train_loader, 'validation': val_loader, 'official_test': test_loader}
loader_counts = {name: len(loader.dataset) for name, loader in loaders.items()}
if loader_counts != EXPECTED_COUNTS:
    raise AssertionError(f'Actual loader counts differ from expected counts: {loader_counts}')

from collections import Counter
class_counts = {}
split_paths = {}
for split, loader in loaders.items():
    counts = Counter(sample['label'] for sample in loader.dataset.samples)
    class_counts[split] = {FER2013_CLASSES[i]: counts[i] for i in range(7)}
    split_paths[split] = {sample['path'] for sample in loader.dataset.samples}
if class_counts != EXPECTED_CLASS_COUNTS:
    raise AssertionError(f'Actual per-class loader counts differ from expected values: {class_counts}')
if split_paths['train'] & split_paths['validation'] or split_paths['train'] & split_paths['official_test'] or split_paths['validation'] & split_paths['official_test']:
    raise AssertionError('Dataset samples leak between splits')
def split_hash(dataset):
    items = sorted((Path(sample['path']).relative_to(DATA_ROOT).as_posix(), int(sample['label'])) for sample in dataset.samples)
    payload = json.dumps(items, separators=(',', ':'), ensure_ascii=True).encode()
    return hashlib.sha256(payload).hexdigest()
SPLIT_MANIFEST_HASHES = {name: split_hash(loader.dataset) for name, loader in loaders.items()}
DATASET_AUDIT = {
    'root': str(DATA_ROOT), 'counts': loader_counts,
    'disk_counts_by_folder': disk_counts, 'loader_counts_by_class': class_counts,
    'class_mapping': FER2013_FOLDER_TO_LABEL,
    'split_manifest_sha256': SPLIT_MANIFEST_HASHES,
    'validation_protocol': 'Seed-42 stratified 15% split of FER2013 train folder',
    'test_protocol': 'Official FER2013 test folder, used only after model selection',
}
(RUN_ROOT / 'dataset_audit.json').write_text(json.dumps(DATASET_AUDIT, indent=2), encoding='utf-8')
print(json.dumps(DATASET_AUDIT, indent=2))

## 8. GPU forward-pass and training-step smoke test

This uses the configured 160×160 input and full batch size 32 in float32, with no automatic mixed precision or reduced batch fallback. It performs only a temporary optimizer step; it does not write or alter any experiment checkpoint.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('CUDA was lost; stop before training.')
set_seed(42)
fixed_config = json.loads(json.dumps(CONFIGS['fixed']))
fixed_config['model']['swin']['pretrained'] = True
smoke_model = build_framework(fixed_config).to(device).train()
smoke_model.freeze_backbone()
smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(), lr=fixed_config['training']['learning_rate'],
    weight_decay=fixed_config['training']['weight_decay'],
    betas=tuple(fixed_config['training']['betas']),
)
class_weights, _ = compute_inverse_frequency_weights(train_loader.dataset, 7)
smoke_loss_fn = torch.nn.CrossEntropyLoss(
    weight=class_weights.to(device), label_smoothing=fixed_config['training']['label_smoothing'],
)
smoke_batch = next(iter(train_loader))
smoke_images = smoke_batch['image'].to(device, non_blocking=True)
smoke_labels = smoke_batch['label'].to(device, non_blocking=True)
if tuple(smoke_images.shape) != (32, 3, 160, 160):
    raise AssertionError(f'Unexpected smoke batch shape {tuple(smoke_images.shape)}')
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize(device)
smoke_start = time.perf_counter()
smoke_optimizer.zero_grad(set_to_none=True)
smoke_output = smoke_model.forward_frame(smoke_images)
smoke_loss = smoke_loss_fn(smoke_output['logits'], smoke_labels)
smoke_loss.backward()
torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), fixed_config['training']['gradient_clip_norm'])
smoke_optimizer.step()
torch.cuda.synchronize(device)
SMOKE_STEP_SECONDS = time.perf_counter() - smoke_start
SMOKE_PEAK_MEMORY_BYTES = int(torch.cuda.max_memory_allocated(device))
print({'batch_shape': tuple(smoke_images.shape), 'loss': float(smoke_loss.item()), 'step_seconds': SMOKE_STEP_SECONDS, 'peak_gpu_memory_bytes': SMOKE_PEAK_MEMORY_BYTES, 'gpu_free_after_step_bytes': torch.cuda.mem_get_info(device)[0]})
del smoke_model, smoke_optimizer, smoke_output, smoke_images, smoke_labels
torch.cuda.empty_cache()

## 9. Training-duration estimate, memory check, and explicit approval gate

Measure several real training steps and validation forwards, including batch loading and host-to-device transfer, to estimate epoch time on this GPU. The estimate excludes checkpoint/Drive I/O, pretrained-weight download, and Colab overhead, so actual wall time may be longer. Verify that frozen and unfrozen steps fit without reducing settings. A small save/load/resume smoke test also checks the mounted Drive filesystem. The next cell will not train unless `APPROVE_TRAINING` is explicitly set to `True` after reviewing the estimate.

In [ ]:
fixed_config = json.loads(json.dumps(CONFIGS['fixed']))
fixed_config['model']['swin']['pretrained'] = False
probe_model = build_framework(fixed_config).to(device).train()
probe_model.freeze_backbone()
probe_optimizer = torch.optim.AdamW(
    probe_model.parameters(), lr=fixed_config['training']['learning_rate'],
    weight_decay=fixed_config['training']['weight_decay'],
    betas=tuple(fixed_config['training']['betas']),
)
probe_loss_fn = torch.nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=fixed_config['training']['label_smoothing'],
)
timed_train = []
train_iter = iter(train_loader)
torch.cuda.reset_peak_memory_stats(device)
for step_index in range(8):
    torch.cuda.synchronize(device); start = time.perf_counter()
    batch = next(train_iter)
    images, labels = batch['image'].to(device), batch['label'].to(device)
    probe_optimizer.zero_grad(set_to_none=True)
    logits = probe_model.forward_frame(images)['logits']
    loss = probe_loss_fn(logits, labels); loss.backward(); probe_optimizer.step()
    torch.cuda.synchronize(device)
    if step_index >= 2: timed_train.append(time.perf_counter() - start)
for parameter in probe_model.spatial_encoder.backbone.parameters(): parameter.requires_grad = True
timed_unfrozen = []
for step_index in range(3):
    torch.cuda.synchronize(device); start = time.perf_counter()
    batch = next(train_iter)
    images, labels = batch['image'].to(device), batch['label'].to(device)
    probe_optimizer.zero_grad(set_to_none=True)
    logits = probe_model.forward_frame(images)['logits']
    loss = probe_loss_fn(logits, labels); loss.backward(); probe_optimizer.step()
    torch.cuda.synchronize(device)
    if step_index >= 1: timed_unfrozen.append(time.perf_counter() - start)
probe_model.eval()
timed_val = []
with torch.inference_mode():
    val_iter = iter(val_loader)
    for step_index in range(3):
        torch.cuda.synchronize(device); start = time.perf_counter()
        batch = next(val_iter)
        images = batch['image'].to(device)
        probe_model.forward_frame(images)
        torch.cuda.synchronize(device); timed_val.append(time.perf_counter() - start)
import math, statistics
batch_size = fixed_config['training']['batch_size']
freeze_epochs = fixed_config['training']['freeze_backbone_epochs']
num_epochs = fixed_config['training']['num_epochs']
batches_per_epoch = math.ceil(EXPECTED_COUNTS['train'] / batch_size)
val_batches = math.ceil(EXPECTED_COUNTS['validation'] / batch_size)
frozen_seconds = statistics.mean(timed_train)
unfrozen_seconds = statistics.mean(timed_unfrozen)
val_seconds = statistics.mean(timed_val)
frozen_epoch_seconds = frozen_seconds * batches_per_epoch + val_seconds * val_batches
unfrozen_epoch_seconds = unfrozen_seconds * batches_per_epoch + val_seconds * val_batches
total_seconds = frozen_epoch_seconds * freeze_epochs + unfrozen_epoch_seconds * max(0, num_epochs - freeze_epochs)
TRAINING_ESTIMATE = {
    'gpu': torch.cuda.get_device_name(device), 'batch_size': batch_size, 'resolution': fixed_config['dataset']['image_size'],
    'train_batches_per_epoch': batches_per_epoch, 'validation_batches': val_batches,
    'frozen_step_seconds_mean': frozen_seconds, 'unfrozen_step_seconds_mean': unfrozen_seconds,
    'validation_forward_seconds_mean': val_seconds,
    'estimated_frozen_epoch_hours': frozen_epoch_seconds / 3600,
    'estimated_epoch_after_unfreeze_hours': unfrozen_epoch_seconds / 3600,
    'estimated_50_epoch_single_variant_hours': total_seconds / 3600,
    'estimated_three_variant_hours': 3 * total_seconds / 3600,
    'estimate_excludes': ['checkpoint and Google Drive I/O', 'pretrained-weight download', 'Colab startup and runtime overhead'],
    'peak_memory_bytes': int(torch.cuda.max_memory_allocated(device)),
    'free_memory_bytes': int(torch.cuda.mem_get_info(device)[0]),
}
(RUN_ROOT / 'environment').mkdir(exist_ok=True)
(RUN_ROOT / 'environment' / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2, default=str), encoding='utf-8')
(RUN_ROOT / 'environment' / 'training_estimate.json').write_text(json.dumps(TRAINING_ESTIMATE, indent=2), encoding='utf-8')
print(json.dumps(TRAINING_ESTIMATE, indent=2))
del probe_model, probe_optimizer
torch.cuda.empty_cache()
if TRAINING_ESTIMATE['free_memory_bytes'] < 512 * 1024 * 1024:
    raise RuntimeError('Less than 512 MiB GPU memory is free after the full smoke/estimate steps; stop without reducing settings.')

import tempfile
from utils.logger import TrainingLogger
with tempfile.TemporaryDirectory(prefix='checkpoint_io_smoke_', dir=RUN_ROOT) as temp_dir:
    smoke_checkpoint_dir = Path(temp_dir) / 'checkpoints'
    smoke_model = torch.nn.Linear(2, 2)
    smoke_optimizer = torch.optim.AdamW(smoke_model.parameters())
    smoke_scheduler = torch.optim.lr_scheduler.StepLR(smoke_optimizer, step_size=1)
    smoke_logger = TrainingLogger(log_dir=temp_dir, experiment_name='drive_checkpoint_smoke', use_tensorboard=False)
    smoke_logger.save_checkpoint(
        smoke_model, smoke_optimizer, epoch=1, metrics={'accuracy': 0.5},
        checkpoint_dir=str(smoke_checkpoint_dir), is_best=True,
        metadata={'region_attention_mode': 'fixed', 'early_stopping': {'best_value': 0.5, 'counter': 0, 'should_stop': False}},
        scheduler=smoke_scheduler, save_epoch_checkpoint=False,
    )
    smoke_logger.close()
    latest = torch.load(smoke_checkpoint_dir / 'latest_model.pth', map_location='cpu', weights_only=False)
    best = torch.load(smoke_checkpoint_dir / 'best_model.pth', map_location='cpu', weights_only=False)
    if latest['epoch'] != 1 or best['epoch'] != 1 or 'scheduler' not in latest or 'rng_state' not in latest:
        raise AssertionError('Drive checkpoint save/load smoke test failed.')
    resumed_model = torch.nn.Linear(2, 2)
    resumed_optimizer = torch.optim.AdamW(resumed_model.parameters())
    resumed_scheduler = torch.optim.lr_scheduler.StepLR(resumed_optimizer, step_size=1)
    resumed_model.load_state_dict(latest['model'], strict=True)
    resumed_optimizer.load_state_dict(latest['optimizer'])
    resumed_scheduler.load_state_dict(latest['scheduler'])
    validate_checkpoint_region_mode(latest, 'fixed', str(smoke_checkpoint_dir / 'latest_model.pth'))
print('Drive checkpoint atomic-save and latest/best load smoke test passed; temporary files removed.')
APPROVE_TRAINING = False  # Review the measured estimate, then change to True to explicitly start long runs.
if not APPROVE_TRAINING:
    print('Pre-training checks complete. No training started. Set APPROVE_TRAINING=True only after reviewing the measured GPU cost.')

## 10. Train `off`

Training runs in a separate variant directory on Drive. Each epoch atomically updates `latest_model.pth`; `best_model.pth` is validation-selected and distinct. To conserve Drive storage, this notebook disables redundant epoch-by-epoch archives for all variants. Existing outputs are never silently overwritten; resume uses only a compatible latest checkpoint from that same variant and resolved configuration.

In [ ]:
import copy
def resolved_drive_config(mode):
    config = copy.deepcopy(CONFIGS[mode])
    config['dataset']['root'] = str(DATA_ROOT)
    config['training']['save_epoch_checkpoints'] = False
    variant_root = RUN_ROOT / mode
    config['paths']['checkpoint_dir'] = str(variant_root / 'checkpoints')
    config['paths']['log_dir'] = str(variant_root / 'logs')
    config['paths']['results_dir'] = str(variant_root / 'results')
    config['model']['swin']['pretrained'] = True
    return config

def train_variant(mode):
    if not APPROVE_TRAINING:
        raise RuntimeError('Training approval is still False.')
    config = resolved_drive_config(mode)
    variant_root = RUN_ROOT / mode
    config_dir = variant_root / 'configs'; config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / f'{mode}_resolved.yaml'
    import yaml
    if config_path.exists():
        prior = yaml.safe_load(config_path.read_text(encoding='utf-8'))
        if prior != config:
            raise RuntimeError(f'Resolved config changed for existing {mode} experiment.')
    else:
        config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    checkpoint_dir = Path(config['paths']['checkpoint_dir'])
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    latest = checkpoint_dir / 'latest_model.pth'
    best = checkpoint_dir / 'best_model.pth'
    resume_path = None
    if latest.exists():
        if not RESUME_EXISTING_RUN:
            raise FileExistsError(f'{latest} exists; set RESUME_EXISTING_RUN=True to resume.')
        saved = torch.load(latest, map_location='cpu', weights_only=False)
        validate_checkpoint_region_mode(saved, mode, str(latest))
        experiment = saved.get('experiment', {})
        if experiment.get('training_config') != config:
            raise RuntimeError(f'Checkpoint config differs from this {mode} run; refusing mixed-configuration resume.')
        if experiment.get('seed') != config['training']['seed']:
            raise RuntimeError(f'Checkpoint seed differs from this {mode} run; refusing mixed-seed resume.')
        resume_path = latest
    elif any(checkpoint_dir.glob('*.pth')):
        raise RuntimeError(f'{checkpoint_dir} contains checkpoint files but no latest_model.pth; inspect manually rather than silently restarting.')
    (variant_root / 'logs').mkdir(parents=True, exist_ok=True)
    log_path = variant_root / 'logs' / 'train_console.log'
    command = [sys.executable, str(PROJECT_DIR / 'scripts/train.py'), '--config', str(config_path)]
    if resume_path is not None:
        command += ['--resume', str(resume_path)]
    with log_path.open('a', encoding='utf-8') as log:
        log.write(f'\nCOMMAND: {command}\n')
        log.flush()
        result = subprocess.run(command, cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        raise RuntimeError(f'{mode} training failed; inspect durable log {log_path}; latest checkpoint is preserved if an epoch completed.')
    if not best.is_file() or not latest.is_file():
        raise RuntimeError(f'{mode} run did not produce both best and latest checkpoints.')
    return {'mode': mode, 'config': str(config_path), 'latest': str(latest), 'best': str(best), 'log': str(log_path)}

off_training = train_variant('off')
print(off_training)

## 11. Train `fixed`

Uses equal one-third eye, eyebrow, and mouth weights; the adaptive weighting MLP is not used in fixed mode. It shares the same data and training protocol and writes only under this run's `fixed` directory.

In [ ]:
fixed_training = train_variant('fixed')
print(fixed_training)

## 12. Train `adaptive`

Uses the existing input-dependent learned region weighting module. This is the current ARWM, not the proposed Context-Aware enhancement.

In [ ]:
adaptive_training = train_variant('adaptive')
print(adaptive_training)

## 13. Final evaluation on the official FER2013 test split

Run only after all three best validation-selected checkpoints exist. The evaluator uses the same deterministic test loader and all 7,178 official test images for each model; it records accuracy, balanced accuracy, macro/weighted F1, per-class metrics, confusion matrices, parameter counts, model-only inference timing, and separate construction/checkpoint/state-dict load times.

In [ ]:
if not APPROVE_TRAINING:
    raise RuntimeError('No training approved; evaluation is not being presented as a new experiment.')
variant_checkpoints = {
    mode: RUN_ROOT / mode / 'checkpoints' / 'best_model.pth'
    for mode in MODES
}
missing = [str(path) for path in variant_checkpoints.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing trained best checkpoints: {missing}')
eval_output = RUN_ROOT / 'evaluation'
eval_command = [
    sys.executable, str(PROJECT_DIR / 'scripts/evaluate_arwm_variants.py'),
    '--config', str(RUN_ROOT / 'fixed/configs/fixed_resolved.yaml'),
    '--off-checkpoint', str(variant_checkpoints['off']),
    '--fixed-checkpoint', str(variant_checkpoints['fixed']),
    '--adaptive-checkpoint', str(variant_checkpoints['adaptive']),
    '--batch-size', '32', '--warmup-batches', '3', '--output', str(eval_output),
]
evaluation_log = RUN_ROOT / 'evaluation' / 'evaluation_console.log'
eval_output.mkdir(parents=True, exist_ok=True)
with evaluation_log.open('w', encoding='utf-8') as log:
    result = subprocess.run(eval_command, cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
if result.returncode:
    raise RuntimeError(f'Full test evaluation failed; see {evaluation_log}. No partial result should be described as final.')
metrics_path = eval_output / 'metrics.json'
report = json.loads(metrics_path.read_text(encoding='utf-8'))
if report['evaluation']['sample_count_in_split'] != 7178:
    raise AssertionError('Official test count changed; do not report this evaluation as the verified split.')
for mode in MODES:
    if report['variants'][mode].get('status') != 'complete_test' or report['variants'][mode].get('number_of_evaluated_samples') != 7178:
        raise AssertionError(f'{mode} did not evaluate exactly all 7,178 test examples')
print('All three variants evaluated on the same official test count:', {m: report['variants'][m]['number_of_evaluated_samples'] for m in MODES})

## 14. Inference latency and parameter-count benchmarking

The evaluation records model-only `forward_frame` latency with CUDA synchronization and three warm-up batches, excluding decoding, preprocessing, and host-to-device input transfer. Latency is reported per batch and amortized per image at batch size 32. Model construction, checkpoint file read, and strict state-dict load are timed separately. Hardware, resolution, sample count, parameter counts, and timing method remain in `metrics.json`.

## 15. Results table, per-variant reports, plots, and scientific comparison

This cell creates the requested comparison JSON/CSV/Markdown, per-variant report JSON/Markdown and confusion-matrix PNGs. Differences from `off` are absolute percentage-point differences, not significance tests. It refuses to produce a comparison if any variant is partial/unavailable. Historical local reports are not included as controlled experiment results.

In [ ]:
import csv, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

variants = report['variants']
if any(variants[m]['status'] != 'complete_test' for m in MODES):
    raise RuntimeError('Cannot create a final three-way comparison from incomplete variants.')
base_metrics = variants['off']['metrics']
comparison = {
    'experiment_type': 'new controlled three-way ARWM ablation',
    'historical_results_included': False,
    'dataset': report['evaluation'],
    'dataset_audit': DATASET_AUDIT,
    'environment': ENVIRONMENT,
    'training_estimate': TRAINING_ESTIMATE,
    'variants': {},
    'interpretation': 'Descriptive results only. No statistical significance or emotion-recognition improvement is claimed without appropriate replicated analysis.',
}
rows = []
for mode in MODES:
    item = variants[mode]
    metrics = item['metrics']
    checkpoint = variant_checkpoints[mode]
    ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
    mode_result = {
        **item,
        'actual_checkpoint_path': str(checkpoint),
        'checkpoint_sha256': sha256_file(checkpoint),
        'selected_epoch': ckpt.get('epoch'),
        'seed': ckpt.get('experiment', {}).get('seed'),
        'resolved_config_path': str(RUN_ROOT / mode / 'configs' / f'{mode}_resolved.yaml'),
        'configuration': resolved_drive_config(mode),
        'training_split_counts': DATASET_AUDIT['counts'],
        'test_sample_count': 7178,
        'input_resolution': 160,
        'batch_size': 32,
        'hardware': ENVIRONMENT,
        'delta_vs_off_percentage_points': {
            'macro_f1': (metrics['macro_f1'] - base_metrics['macro_f1']) * 100.0,
            'balanced_accuracy': (metrics['balanced_accuracy'] - base_metrics['balanced_accuracy']) * 100.0,
        },
    }
    comparison['variants'][mode] = mode_result
    mode_dir = eval_output / mode
    mode_dir.mkdir(parents=True, exist_ok=True)
    (mode_dir / 'evaluation_report.json').write_text(json.dumps(mode_result, indent=2, default=str), encoding='utf-8')
    matrix = np.asarray(item['confusion_matrix'], dtype=int)
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(7), FER2013_CLASSES, rotation=45, ha='right')
    ax.set_yticks(range(7), FER2013_CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'{mode}: official test confusion matrix')
    fig.colorbar(image, ax=ax)
    fig.tight_layout(); fig.savefig(mode_dir / 'confusion_matrix.png', dpi=160); plt.close(fig)
    with (mode_dir / 'evaluation_report.md').open('w', encoding='utf-8') as stream:
        stream.write(f"# {mode.title()} ARWM evaluation\n\n")
        stream.write(f"- Checkpoint: `{checkpoint}` (epoch {ckpt.get('epoch')})\n- Seed: {mode_result['seed']}\n- Official test samples: 7,178\n- Device: {ENVIRONMENT['gpu_name']}\n- Batch/resolution: 32 / 160x160\n- Model load: {item['timing']['model_and_checkpoint_load_seconds']:.3f}s\n- Mean model latency: {item['timing']['mean_latency_ms_per_image']:.3f} ms/image\n\n")
        stream.write('| Class | Precision | Recall | F1 |\n|---|---:|---:|---:|\n')
        for name, class_metrics in metrics['per_class'].items():
            stream.write(f"| {name} | {class_metrics['precision']:.4f} | {class_metrics['recall']:.4f} | {class_metrics['f1']:.4f} |\n")
    rows.append({
        'mode': mode, 'checkpoint': str(checkpoint), 'checkpoint_sha256': mode_result['checkpoint_sha256'],
        'epoch': ckpt.get('epoch'), 'seed': mode_result['seed'], 'test_samples': 7178,
        'accuracy': metrics['accuracy'], 'balanced_accuracy': metrics['balanced_accuracy'],
        'macro_f1': metrics['macro_f1'], 'weighted_f1': metrics['weighted_f1'],
        'macro_f1_delta_pp_vs_off': mode_result['delta_vs_off_percentage_points']['macro_f1'],
        'balanced_accuracy_delta_pp_vs_off': mode_result['delta_vs_off_percentage_points']['balanced_accuracy'],
        'trainable_parameters': item['trainable_parameters'], 'total_parameters': item['total_parameters'],
        'mean_latency_ms_per_image': item['timing']['mean_latency_ms_per_image'],
        'model_and_checkpoint_load_seconds': item['timing']['model_and_checkpoint_load_seconds'],
    })
(eval_output / 'arwm_comparison.json').write_text(json.dumps(comparison, indent=2, default=str), encoding='utf-8')
with (eval_output / 'arwm_comparison.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
summary = ['# Controlled ARWM comparison', '', '- Dataset: FER2013 official test; 7,178 examples per variant.', '- The test split was not used for selection.', '- Historical reports are excluded.', '- Differences are descriptive percentage points; no statistical significance is claimed.', '', '| Variant | Accuracy | Balanced accuracy | Macro-F1 | Weighted-F1 | Macro-F1 delta (pp) | Balanced accuracy delta (pp) | Params | ms/image |', '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for row in rows:
    summary.append(f"| {row['mode']} | {row['accuracy']:.4f} | {row['balanced_accuracy']:.4f} | {row['macro_f1']:.4f} | {row['weighted_f1']:.4f} | {row['macro_f1_delta_pp_vs_off']:+.2f} | {row['balanced_accuracy_delta_pp_vs_off']:+.2f} | {row['total_parameters']:,} | {row['mean_latency_ms_per_image']:.3f} |")
(eval_output / 'arwm_experiment_report.md').write_text('\n'.join(summary) + '\n', encoding='utf-8')
print('\n'.join(summary))

## 16. Export, integrity verification, and return to the Windows project

All checkpoints, resolved YAMLs, logs, environment/dataset audits, JSON/CSV metrics, per-variant reports, and confusion plots are written directly to the persistent Drive folder printed below. The original checkpoint hashes are checked again after the experiment. Download the `experiments/<RUN_ID>` folder from Drive, then copy results into a **new** local output folder; do not replace the original checkpoints.

In [ ]:
CHECKPOINT_HASHES_AFTER = {name: sha256_file(path) for name, path in ORIGINAL_CHECKPOINTS.items()}
if CHECKPOINT_HASHES_AFTER != CHECKPOINT_HASHES_BEFORE:
    raise RuntimeError(f'Original checkpoints changed in the Colab source copy: before={CHECKPOINT_HASHES_BEFORE}, after={CHECKPOINT_HASHES_AFTER}')
integrity = {
    'original_checkpoint_hashes_before': CHECKPOINT_HASHES_BEFORE,
    'original_checkpoint_hashes_after': CHECKPOINT_HASHES_AFTER,
    'unchanged': CHECKPOINT_HASHES_BEFORE == CHECKPOINT_HASHES_AFTER,
    'run_id': RUN_ID,
    'run_root': str(RUN_ROOT),
    'project_backup_zip': str(archive_source),
    'evaluation_status': 'complete_three_way' if (RUN_ROOT / 'evaluation/arwm_comparison.json').is_file() else 'not_completed',
}
(RUN_ROOT / 'integrity_manifest.json').write_text(json.dumps(integrity, indent=2), encoding='utf-8')
print(json.dumps(integrity, indent=2))
print('\nPersistent experiment folder:', RUN_ROOT)
print('In Google Drive: MyDrive/SWIN_ARWM_Colab/experiments/' + RUN_ID)
print('To return results to Windows, download that Drive folder and copy its contents into a fresh folder such as C:\\Users\\AMSV\\OneDrive\\Desktop\\SWIN\\results\\arwm\\colab_' + RUN_ID + '. Do not copy over checkpoints\\fer2013_arwm or checkpoints\\fer2013_weighted_freeze3_lr1e4.')